<img style="float: center;" src='./assets/jwebbinar.png' width="1000px"/> 

## JWebbinar 49
_June 25, 2026_
## JHAT Examples

### Author: Justin Pierel (STScI), Armin Rest (STScI)

**Purpose**:<BR>
This notebook has a couple of common use cases for JWST alignment, using the JHAT package. First is aligning data to an external catalog, and second is aligning between instruments (MIRI to NIRCam). 

**Data**:<BR>
This example is set up to use an example dataset from the JADES program, followed by some early JWST data from an ERS program 2727 covering the Supernova 2021afdx. 


<hr style="border:1px solid gray"> </hr>

## Table of Contents
1. Aligning to an external catalog
2. Aligning MIRI to NIRCam
    


# 1. Section 1 - Align to catalog

<hr style="border:1px solid gray"> </hr>

In [ ]:
# Package imports
from jwst.datamodels import ImageModel
import re,os
import jhat
import matplotlib.pyplot as plt
from astropy.table import Table
import os

## If you have a CRDS issue, try uncommenting the following three lines
# os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache/jwst")
# os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# os.makedirs(os.environ["CRDS_PATH"], exist_ok=True)

##### Let's start with creating our catalog from an image, i.e., a relative alignment. You could also bring in this catalog from any external source as long as it has the key column names of ra, dec, mag, and magerr. 

In [ ]:
input_image='../data/jw01180013001_10101_00005_nrcb2_cal.fits'
outrootdir = '../data/aligned'
outsubdir = 'cat_example'
telescope = 'jwst'
reference_image = '../data/jades_template_example_i2d.fits'

wcs_align = jhat.st_wcs_align()
verbose=2
wcs_align.verbose=verbose

# first rough cut: best d_rotated+-rough_cut_pix. This is the upper limit for rough_cut
wcs_align.rough_cut_px_min = wcs_align.rough_cut_px_max = 1.5


In [ ]:
wcs_align.set_outbasename(outrootdir=outrootdir,outsubdir=outsubdir,inputname=reference_image)
# set the telescope
wcs_align.set_telescope(telescope=telescope,imname=reference_image)


In [ ]:
wcs_align.phot.verbose = wcs_align.verbose
refcatfilename = f'{wcs_align.outbasename}.phot.txt'
(refcatfilename_4check,refcat_loaded) = wcs_align.phot.run_phot(reference_image,
                                                          use_dq=True,
                                                          photfilename=refcatfilename,
                                                          overwrite=True)

In [ ]:
refcat = Table.read(refcatfilename,format='ascii')
refcat

##### Some of the sources in the catalog will be junk, or faint diffuse galaxies. A simple check is to make a magnitude vs. sharpness plot and identify cuts to make that localize the stellar locus. 

In [ ]:
fig,ax = plt.subplots(1,1)
ax.scatter(refcat['sharpness'],refcat['mag'],color='k')
ax.set_ylabel('AB Mag')
ax.set_xlabel('Sharpness')
plt.show()

In [ ]:
fig,ax = plt.subplots(1,1)
magupper = 26
ax.scatter(refcat[refcat['mag']<magupper]['sharpness'],refcat[refcat['mag']<magupper]['mag'],color='k')
plt.show()

In [ ]:
wcs_align.set_outbasename(outrootdir=outrootdir,outsubdir=outsubdir,inputname=input_image)
# set the telescope
wcs_align.set_telescope(telescope=telescope,imname=input_image)


In [ ]:
# do the photometry on the target image
xshift=0.0 
yshift=0.0 # This can be estimated if necessary by, e.g., comparing images in DS9 or identifying the stellar locus in later figures
wcs_align.phot.verbose = wcs_align.verbose
photfilename = f'{wcs_align.outbasename}.phot.txt'
(photfilename_4check,photcat_loaded) = wcs_align.phot.run_phot(input_image,
                                                          use_dq=True,
                                                          photfilename=photfilename,
                                                          xshift=xshift,
                                                          yshift=yshift,
                                                          overwrite=True)
# get the indices to the detections
ixs = wcs_align.phot.getindices()


##### Now we match the target image catalog to the reference catalog we created above. The next two cells do the matching and show the process in a series of figures. 

In [ ]:
sharpness_lim=(None,0.9)
roundness1_lim=(-0.75,0.75)
objmag_lim=(17,26)
Nbright=None
refmag_lim = (None,26)
dmag_max = None

# make the initial cut on the image photometry catalog on magnitudes, sharpness, roundness etc
ixs_use = wcs_align.phot.initial_cut_photcat(dmag_max = dmag_max,
                                        sharpness_lim = sharpness_lim, # sharpness limits
                                        roundness1_lim = roundness1_lim, # roundness1 limits 
                                        objmag_lim = objmag_lim, # limits on mag, the magnitude of the objects in the image
                                        Nbright = Nbright,
                                        ixs=ixs)

wcs_align.phot.load_and_match_refcat(ixs_obj=ixs_use,
                                refcatname=refcatfilename,
                                refcat_magcol='mag',
                                refcat_magerrcol='dmag',
                                refcat_racol='ra',
                                refcat_deccol='dec',
                                refmag_lim=refmag_lim, # limits for initial cut
                                refmagerr_lim=refmag_lim, # limits for initial cut, needs to be added to options
                                refcolor_lim=(None,None), # limits for initial cut, needs to be added to options
                                )

# Save the refcat entries!
refcatfilename = f'{wcs_align.outbasename}.refcat.txt'
print(f'Saving refcat file into {refcatfilename}')
wcs_align.phot.refcat.write(refcatfilename,overwrite=True)



In [ ]:
# maximum distance between source in image and refcat object, in arcsec 
d2d_max = None
# maximum uncertainty
dmag_max = None
# limits on color of image mag and reference color magnitude
delta_mag_lim =(None,None)
slope_min=-0.008
slope_Nsteps = 20 # slope_max=-slope_min, slope_stepsize=(slope_max-slope_min)/slope_Nsteps
Nfwhm = 2.5
show_initial_plot=1
show_histofit_plots=2
savephottable=True
outbasename=wcs_align.outbasename


ixs_bestmatch= wcs_align.find_good_refcat_matches(ixs=ixs_use,
                                             d2d_max = d2d_max,
                                             delta_mag_lim = delta_mag_lim, # limits on mag-refcat_mainfilter
                                             refmag_lim = refmag_lim, # limits on refcat_mainfilter, the magnitude of the reference catalog
                                             slope_min=slope_min, 
                                             slope_Nsteps = slope_Nsteps, # slope_max=-slope_min, slope_stepsize=(slope_max-slope_min)/slope_Nsteps
                                             show_initial_plot=show_initial_plot,
                                             show_histofit_plots=show_histofit_plots,
                                             savephottable=savephottable,
                                             outbasename=outbasename
                                             )     


##### Now we actually apply the matches and correct the WCS of the target image, and show some summary plots. 

In [ ]:
showplots = 1
saveplots = 1
savephottable=1
jhatfits = f'{wcs_align.outbasename}_jhat.fits'
(runflag,jhatfits) = wcs_align.run_align2refcat(input_image,
                                           outputfits=jhatfits,
                                           ixs=ixs_bestmatch,
                                           overwrite=True)

#if self.telescope.lower()=='jwst':
wcs_align.update_phottable_final_wcs(jhatfits,
                                    ixs_bestmatch = ixs_bestmatch,
                                    showplots=showplots,
                                    saveplots=saveplots,
                                    savephottable=savephottable,
                                    overwrite=True
                                    )


# Section 2: Aligning MIRI to NIRCam

Now you try! There is a solution notebook in this directory if you get stuck, but use the above example to try and obtain a good alignment between MIRI and NIRCam images below.

In [ ]:
input_image = '../data/jw02727007001_02101_00003_mirimage_cal.fits' # F770W 
#input_image = '../data/jw02727007001_02107_00002_mirimage_cal.fits' # F1800W
outrootdir = '../data/aligned'
outsubdir = 'cat_example'
telescope = 'jwst'
reference_image = '../data/jw02727-o002_t062_nircam_clear-f444w_i2d.fits'

wcs_align = jhat.st_wcs_align()
verbose=2
wcs_align.verbose=verbose

